### Week 8 — Sentiment Analysis
1. Task:
   - Use a reviews dataset.

2. Steps:
   - Preprocess text
   - Train sentiment model
   - Predict sentiment for new reviews

3. Example predictions:

    - "Product is amazing" → Positive

    - "Worst experience" → Negative

### 1.Importing Libraries

In [ ]:
import re
import string

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

pd.set_option("display.max_colwidth", 100)
print("Libraries imported successfully.")


Libraries imported successfully.


### 2.Loading Dataset

In [10]:
df = pd.read_csv('kindle_review.csv')
print("Dataset Shape:",df.shape)
df.head()

Dataset Shape: (12000, 11)


,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to mess with, as the man who was just hauled out of t...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it down so I read it all in one sitting. The sex scenes...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four books so I wasn't expecting it to &#34;conclude&#...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketbooks instead of writing them. While this light murde...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in library was pleased to find it price was right,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Unnamed: 0.1    12000 non-null  int64
 1   Unnamed: 0      12000 non-null  int64
 2   asin            12000 non-null  str  
 3   helpful         12000 non-null  str  
 4   rating          12000 non-null  int64
 5   reviewText      12000 non-null  str  
 6   reviewTime      12000 non-null  str  
 7   reviewerID      12000 non-null  str  
 8   reviewerName    11962 non-null  str  
 9   summary         11998 non-null  str  
 10  unixReviewTime  12000 non-null  int64
dtypes: int64(4), str(7)
memory usage: 1.0 MB


### 3.Giving Labels For the Star Rating:
* **rating > 3** (4 or 5 stars) → `Positive`
* **rating <= 3** (1, 2, or 3 stars) → `Negative`

In [11]:
df = df.dropna(subset=['reviewText']).reset_index(drop=True)

df['sentiment'] = df['rating'].apply(lambda r:'Positive'if r >3 else "Negative")
df = df.rename(columns={'reviewText':'review'})
df.insert(0,'review_id',range(1,len(df)+1))

print('cleaned datset shape:',df.shape)
df[['review_id','review','rating','sentiment']].head()

cleaned datset shape: (12000, 13)


,review_id,review,rating,sentiment
0,1,"Jace Rankin may be short, but he's nothing to mess with, as the man who was just hauled out of t...",3,Negative
1,2,Great short read. I didn't want to put it down so I read it all in one sitting. The sex scenes...,5,Positive
2,3,I'll start by saying this is the first of four books so I wasn't expecting it to &#34;conclude&#...,3,Negative
3,4,Aggie is Angela Lansbury who carries pocketbooks instead of writing them. While this light murde...,3,Negative
4,5,I did not expect this type of book to be in library was pleased to find it price was right,4,Positive
